# 07 — Démonstration Hackathon Citiz : CA de La Rochelle

Ce notebook **relit** les tables Gold et les exports ; il ne recalcule aucun résultat. Il sert à montrer, en un seul endroit et dans l'ordre de la soutenance : le diagnostic, les candidats, la recommandation, le prix de l'équité et les contrôles.

**Prérequis :** les scripts du pipeline ont tourné (README, « Reproduire »), au moins jusqu'à `scripts/build_exports.py`.

**Rappel :** le score K10 est un **score de potentiel**, pas une prédiction de fréquentation. Coûts et budgets sont des hypothèses de simulation ou des estimations publiques.

In [ ]:
import json
import sys
from pathlib import Path

import geopandas as gpd
import matplotlib.pyplot as plt
import pandas as pd

RACINE = Path.cwd().resolve()
while not (RACINE / "src").exists() and RACINE != RACINE.parent:
    RACINE = RACINE.parent
sys.path.insert(0, str(RACINE))
from src.export import mesures as m  # mêmes mesures que l'application

EXPORTS = RACINE / "data" / "gold" / "exports"
T = "larochelle"
d = {f.name.removesuffix(f"_{T}.parquet"): pd.read_parquet(f) for f in sorted(EXPORTS.glob(f"*_{T}.parquet"))}
pd.set_option("display.float_format", lambda x: f"{x:,.2f}".replace(",", " "))
ent = lambda x: f"{x:,.0f}".replace(",", " ")           # 18 816
pct = lambda x: f"{100 * x:.1f}".replace(".", ",") + " %"     # 9,8 %
print(f"{len(d)} tables d'export lues depuis {EXPORTS.relative_to(RACINE)}")

## 1. Traçabilité

Chaque table Gold porte un `run_id` : date UTC et commit Git. Un `run_id` finissant par `-modifie` signale du code non commité.

In [ ]:
meta = d["meta_execution"][["dossier", "table", "run_id", "modifie"]]
print("Tables produites par du code non commité :", int(meta["modifie"].sum()))
meta.sort_values(["dossier", "table"])

## 2. Diagnostic : qui a une option à 5 minutes à pied ?

5 minutes = 400 m sur le réseau piéton OpenStreetMap. Un habitant desservi par plusieurs modes compte une fois en « tous modes ».

In [ ]:
k = d["fact_kpi_territoire"].merge(d["dim_mode"], on="mode").sort_values("ordre")
k[["libelle", "pop_totale", "pop_desservie", "pop_non_desservie", "taux_couverture", "options_moyennes_par_habitant"]]

In [ ]:
communes = d["dim_commune"].sort_values("taux_tous_modes")
print("Communes prioritaires (couverture tous modes < 60 %) :", int(communes["prioritaire"].sum()), "sur", len(communes))
communes[communes["prioritaire"]][["nom", "pop_filosofi", "taux_tous_modes", "taux_citiz"]]

In [ ]:
# Objectif 2 du sujet : habitants à 5 min de chaque station ou arrêt (ne s'additionne pas entre stations)
d["dim_offre"].sort_values("pop_5min", ascending=False)[["nom", "mode", "pop_5min", "pop_exclusive"]].head(10)

In [ ]:
# Carte statique : sert aussi de secours si l'application ne s'affiche pas
carreaux = gpd.read_file(EXPORTS / f"carreaux_{T}.geojson")
contours = gpd.read_file(EXPORTS / f"communes_{T}.geojson")
etat = d["fact_acces_mode"].query("mode == 'tous'")[["cell_id", "desservi"]]
carreaux = carreaux.merge(etat, on="cell_id")
ax = carreaux.plot(color=carreaux["desservi"].map({True: "#0072B2", False: "#D55E00"}), figsize=(9, 9), linewidth=0)
contours.boundary.plot(ax=ax, color=contours["prioritaire"].map({True: "#6A3D9A", False: "#9AA3AD"}),
                       linewidth=contours["prioritaire"].map({True: 2.0, False: 0.6}))
ax.set_title("Carreaux de 200 m desservis (bleu) ou non (orange) à 5 min, tous modes\ncontour violet : commune prioritaire")
ax.set_axis_off()
sortie = RACINE / "reports" / "figures" / "carte_couverture_tous_modes.png"
sortie.parent.mkdir(parents=True, exist_ok=True)
plt.savefig(sortie, dpi=150, bbox_inches="tight")
print("Figure :", sortie.relative_to(RACINE))

## 3. Candidats et score de potentiel (K10)

2 200 sites Citiz retenus. La note (sur 100) est la somme de 5 contributions. Elle **départage** ; elle ne décide pas.

In [ ]:
cand = d["dim_candidat"].merge(d["dim_commune"][["commune_insee", "nom"]], on="commune_insee", how="left")
cand.nsmallest(10, "rang")[["rang", "nom", "score", "pop_nouvelle_isolee", "taux_redondance", "frequence_top_n"]]

In [ ]:
premier = cand.nsmallest(1, "rang")["candidat_id"].iloc[0]
d["fact_contribution"].query("candidat_id == @premier").sort_values("ordre")[["libelle_critere", "valeur_normalisee", "contribution"]]

In [ ]:
# Sensibilité : chaque poids × 0,5 ou × 1,5
d["fact_sensibilite"][["variable", "facteur", "spearman", "recouvrement_top_n"]]

## 4. Optimisation sous budget

Couverture maximale sous budget (PuLP / CBC), chaque habitant compté une fois, optimum prouvé à 0,1 % près. 3 budgets × 3 coûts par station, plus 3 variantes équité au coût central.

In [ ]:
sc = d["dim_scenario"].sort_values(["ordre_budget", "cout_station", "variante"])
sc[["libelle", "n_stations", "k6", "k7", "k8", "couverture_citiz_apres", "k12_part_prioritaire", "statut_solution"]]

In [ ]:
ref = m.reference(d["dim_scenario"])
print(f"Référence : {ref['libelle']}")
print(f"  {ref['n_stations']} stations, {ent(ref['k6'])} habitants nouvellement couverts, {ref['k8']:.2f} € par habitant".replace(".", ","))
print(f"  Couverture Citiz : {pct(ref['couverture_citiz_avant'])} → {pct(ref['couverture_citiz_apres'])}")
sel = d["fact_selection"].query("scenario_id == @ref.scenario_id").merge(d["dim_commune"][["commune_insee", "nom"]], on="commune_insee")
sel.sort_values("ordre_ajout")[["ordre_ajout", "nom", "score", "pop_nouvelle_marginale", "pop_nouvelle_cumulee", "cout_cumule"]]

In [ ]:
# Prix de l'équité : habitants perdus à budget égal quand 30 % des gains vont aux communes prioritaires
eq = d["dim_scenario"].query("variante == 'equite'").sort_values("budget")
eq[["budget", "k6", "prix_equite_habitants", "prix_equite_part", "pop_gagnee_prioritaire", "pop_sans_option_gagnee", "k8"]]

In [ ]:
courbe = d["fact_courbe"]
ax = courbe.plot(x="n_stations", y="k6", marker="o", color="#0072B2", legend=False, figsize=(8, 4))
ax.set_xlabel("Stations financées (coût central)")
ax.set_ylabel("Habitants nouvellement couverts")
ax.set_title(f"Rendements décroissants : la 1re station couvre {ent(courbe['gain_marginal'].iloc[0])} habitants, "
             f"la 30e {ent(courbe['gain_marginal'].iloc[-1])}")
plt.savefig(RACINE / "reports" / "figures" / "courbe_budget_impact.png", dpi=150, bbox_inches="tight")

## 5. Contrôles

Les mêmes recoupements que la page 5 de l'application. Chaque écart doit valoir 0 ; la cellule s'arrête sinon.

In [ ]:
ctrl = m.controles(d["fact_acces_mode"], d["dim_cellule"], d["fact_kpi_territoire"], d["dim_scenario"],
                   d["fact_cellules_gagnees"], d["fact_selection"])
assert ctrl["ok"].all(), ctrl[~ctrl["ok"]]
q = d["fact_qualite"]["statut"].value_counts()
print(f"Recoupements : {int(ctrl['ok'].sum())} / {len(ctrl)} à zéro ; qualité : {q.get('ok', 0)} ok, {q.get('alerte', 0)} alertes, {q.get('info', 0)} info")
ctrl

## 6. Limites (à dire, pas à cacher)

- Pas de prédiction d'usage : aucun historique de locations public. Le modèle H6 est conçu, non livré (`docs/model_card.md`).
- Coûts et budgets : hypothèses étiquetées (`docs/registre_couts.md`).
- Seul Citiz est optimisé ; les sites bus et vélo sont scorés.
- Vie privée : aucune donnée personnelle ; carreaux imputés par l'INSEE au titre du secret statistique.